In [96]:
import pandas as pd

In [97]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
dataset_avis =  pd.read_csv("dataset_avis.csv")
dataset_avis

In [99]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report



In [100]:
X = dataset_avis['clean_comment']  # texte déjà nettoyé
y = dataset_avis[['qualité produit','service livraison','service client']].values  # labels binaires

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multiclass import OneVsRestClassifier
from imblearn.over_sampling import SMOTE

# Split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Vectorisation
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# KNN multilabel
knn = KNeighborsClassifier(
    n_neighbors=10,
    metric="cosine",
    weights="distance"   # recommandé pour du texte
)

clf = OneVsRestClassifier(knn)
clf.fit(X_train, y_train)

# Prédictions
y_pred = clf.predict(X_test)
y_pred

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

label_names = ['qualité produit','service livraison','service client']  # adapte à ton cas

for i, label in enumerate(label_names):
    cm = confusion_matrix(y_test[:, i], y_pred[:, i])
    cm_df = pd.DataFrame(
        cm,
        index=['Vrai 0', 'Vrai 1'],
        columns=['Prédit 0', 'Prédit 1']
    )
    
    print(f"\nMatrice de confusion pour {label}")
    print(cm_df)


In [103]:
# for i, label in enumerate(label_names):
#     cm = confusion_matrix(
#         y_test[:, i],
#         y_pred[:, i],
#         labels=[0, 1]
#     )

#     cm_df = pd.DataFrame(
#         cm,
#         index=['Vrai 0', 'Vrai 1'],
#         columns=['Prédit 0', 'Prédit 1']
#     )

#     print(f"\nMatrice de confusion pour {label}")
#     print(cm_df)


In [ ]:
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


# Calcul des matrices de confusion par label
mcm = multilabel_confusion_matrix(y_test, y_pred)

# Affichage avec ConfusionMatrixDisplay
for i, label in enumerate(label_names):
    disp = ConfusionMatrixDisplay(confusion_matrix=mcm[i],
                                  display_labels=[f'{label} 0', f'{label} 1'])
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f"Matrice de confusion - {label}")
    plt.show()

In [ ]:
# Classification report
report = classification_report(
    y_test,
    y_pred,
    target_names=label_names,
    zero_division=0  # évite les warnings si un label n'est pas prédit
)

print(report)

In [106]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multiclass import OneVsRestClassifier

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', OneVsRestClassifier(
        KNeighborsClassifier(
            n_neighbors=10,
            metric='cosine'
        )
    ))
])


In [107]:
from sklearn.metrics import make_scorer, hamming_loss

hamming_scorer = make_scorer(
    hamming_loss,
    greater_is_better=False
)


In [108]:
from sklearn.metrics import make_scorer, hamming_loss

hamming_scorer = make_scorer(
    hamming_loss,
    greater_is_better=False
)


In [109]:
from sklearn.model_selection import cross_validate

scoring = {
    'f1_micro': 'f1_micro',
    'f1_macro': 'f1_macro',
    'hamming_loss': hamming_scorer   # ✅ PAS hamming_loss
}

cv_results = cross_validate(
    pipeline,
    X,
    y,
    cv=5,
    scoring=scoring,
    n_jobs=-1
)


In [ ]:
import numpy as np

print("F1 micro :", np.mean(cv_results['test_f1_micro']))
print("F1 macro :", np.mean(cv_results['test_f1_macro']))
print("Hamming loss :", -np.mean(cv_results['test_hamming_loss']))


In [ ]:
dataset_avis =  pd.read_csv("100_avis_annote.csv", sep=';')
dataset_avis

In [112]:
X_annot = dataset_avis['clean_comment']  # texte déjà nettoyé
y_annot = dataset_avis[['qualité produit','service livraison','service client']].values  # labels binaires

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.multioutput import MultiOutputClassifier

# Vectorisation
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_annot)

clf = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=5))
y_pred = cross_val_predict(clf, X_train, y_annot, cv=5)


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

label_names = ['qualité produit','service livraison','service client']  # adapte à ton cas

for i, label in enumerate(label_names):
    cm = confusion_matrix(y_annot[:, i], y_pred[:, i])
    cm_df = pd.DataFrame(
        cm,
        index=['Vrai 0', 'Vrai 1'],
        columns=['Prédit 0', 'Prédit 1']
    )
    
    print(f"\nMatrice de confusion pour {label}")
    print(cm_df)

In [ ]:
# Classification report
report = classification_report(
    y_annot,
    y_pred,
    target_names=label_names,
    zero_division=0  # évite les warnings si un label n'est pas prédit
)

print(report)